In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)


In [2]:
# ---------- Dataset 1: messy retail orders (main dataset for Sections A & B) ----------
n = 500
df = pd.DataFrame(
  {
"order_id": range(1001, 1001 + n),
"customer_name": np.random.choice(
["Ali Khan", "sara ahmed", "JOHN SMITH", "Fatima Noor", np.nan, "Ali Khan "], n),
"order_date": pd.date_range("2024-01-01", periods=n, freq="6h").astype(str),
"category": np.random.choice(
["Electronics", "Clothing", "Grocery", "Toys", None], n, p=[0.3, 0.25, 0.25, 0.15, 0.05]),
"quantity": np.random.randint(1, 10, n).astype(float),
"unit_price": np.round(np.random.uniform(5, 500, n), 2),
"region": np.random.choice(["North", "South", "East", "West"], n),
"payment_method": np.random.choice(["Card", "Cash", "Card", "Online", None], n),
}
)
df = pd.concat([df, df.sample(15, random_state=1)], ignore_index=True) # inject duplicates
df.loc[df.sample(20, random_state=2).index, "quantity"] = np.nan # inject missing values
df.loc[df.sample(15, random_state=3).index, "unit_price"] = np.nan


In [3]:
df

,order_id,customer_name,order_date,category,quantity,unit_price,region,payment_method
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East,Cash
1,1002,nan,2024-01-01 06:00:00,Clothing,NaN,21.44,North,NaN
2,1003,JOHN SMITH,2024-01-01 12:00:00,Clothing,4.0,43.90,West,Online
3,1004,nan,2024-01-01 18:00:00,Grocery,9.0,201.48,East,Cash
4,1005,nan,2024-01-02 00:00:00,Grocery,1.0,70.69,West,Card
...,...,...,...,...,...,...,...,...
510,1323,Ali Khan,2024-03-21 12:00:00,Electronics,4.0,272.31,East,Card
511,1169,Fatima Noor,2024-02-12 00:00:00,Grocery,2.0,465.82,East,Cash
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West,Cash
513,1067,sara ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South,Card


# **SECTION A — EASY (Q1–Q15)**


In [8]:
# First-look inspection and basic cleaning on df .
# Q1. First look at the data
# Load df and display its shape, the first 5 rows, and the output of .info() .
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        515 non-null    int64  
 1   customer_name   515 non-null    str    
 2   order_date      515 non-null    str    
 3   category        489 non-null    str    
 4   quantity        495 non-null    float64
 5   unit_price      500 non-null    float64
 6   region          515 non-null    str    
 7   payment_method  419 non-null    str    
dtypes: float64(2), int64(1), str(5)
memory usage: 32.3 KB


In [9]:
# Q2. Summary statistics
# Use .describe() to get summary statistics for all numeric columns in df . Then get
# summary statistics for the categorical columns too.
df.describe()

,order_id,quantity,unit_price
count,515.000000,495.000000,500.000000
mean,1249.800000,4.923232,254.871520
std,144.634925,2.678102,144.172621
min,1001.000000,1.000000,7.450000
25%,1123.500000,2.000000,122.580000
50%,1250.000000,5.000000,260.240000
75%,1373.500000,7.000000,370.125000
max,1500.000000,9.000000,499.710000


In [ ]:
# summary of categorical columns # --> include = str
df.describe(include="str")

,customer_name,order_date,category,region,payment_method
count,515,515,489,515,419
unique,6,500,4,4,3
top,Fatima Noor,2024-01-08 18:00:00,Electronics,West,Card
freq,103,2,151,139,220


In [ ]:
# Q3. Count missing values
# Count how many missing values exist in each column of df , and express it as a
# percentage of total rows
count_miss = df.isnull().sum()
percent_miss = np.round((count_miss/df.shape[0]) * 100,2)
miss = pd.DataFrame({"count_miss":count_miss,"percent_miss":percent_miss})
miss

,count_miss,percent_miss
order_id,0,0.00
customer_name,0,0.00
order_date,0,0.00
category,26,5.05
quantity,20,3.88
unit_price,15,2.91
region,0,0.00
payment_method,96,18.64


In [19]:
# Q4. Count duplicate rows
# Find how many fully duplicated rows exist in df 
df.duplicated().sum()

np.int64(15)

In [25]:
# Q5. Fix the date column's type
# order_date is currently a string. Convert it to a proper datetime64 column.
df["order_date"] = pd.to_datetime(df["order_date"])
df["order_date"].dtype


dtype('<M8[us]')

In [40]:
df

,order_id,customer_name,order_date,category,quantity,unit_price,region,payment_method
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East,Cash
1,1002,nan,2024-01-01 06:00:00,Clothing,NaN,21.44,North,NaN
2,1003,JOHN SMITH,2024-01-01 12:00:00,Clothing,4.0,43.90,West,Online
3,1004,nan,2024-01-01 18:00:00,Grocery,9.0,201.48,East,Cash
4,1005,nan,2024-01-02 00:00:00,Grocery,1.0,70.69,West,Card
...,...,...,...,...,...,...,...,...
510,1323,Ali Khan,2024-03-21 12:00:00,Electronics,4.0,272.31,East,Card
511,1169,Fatima Noor,2024-02-12 00:00:00,Grocery,2.0,465.82,East,Cash
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West,Cash
513,1067,sara ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South,Card


In [81]:
# Q6. Standardize a text column
# customer_name has inconsistent casing and stray whitespace ( "sara ahmed" , "JOHN SMITH" ,
# "Ali Khan " ). Clean it so every name is trimmed and in title case.
df = df[df["customer"] != "nan"]
df["customer"] = df["customer"].str.strip().str.title()
df


,order_id,customer,order_date,category,quantity,price,region,payment_method
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East,Cash
2,1003,John Smith,2024-01-01 12:00:00,Clothing,4.0,43.90,West,Online
5,1006,Sara Ahmed,2024-01-02 06:00:00,Electronics,4.0,285.93,North,Card
6,1007,John Smith,2024-01-02 12:00:00,Clothing,6.0,346.29,East,Card
7,1008,John Smith,2024-01-02 18:00:00,Grocery,6.0,401.29,North,NaN
...,...,...,...,...,...,...,...,...
510,1323,Ali Khan,2024-03-21 12:00:00,Electronics,4.0,272.31,East,Card
511,1169,Fatima Noor,2024-02-12 00:00:00,Grocery,2.0,465.82,East,Cash
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West,Cash
513,1067,Sara Ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South,Card


In [66]:
# Q7. Rename columns
# Rename customer_name to customer and unit_price to price .
df = df.rename(columns = {"customer_name":"customer","unit_price": "price"})
df

,order_id,customer,order_date,category,quantity,price,region,payment_method
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East,Cash
2,1003,John Smith,2024-01-01 12:00:00,Clothing,4.0,43.90,West,Online
5,1006,Sara Ahmed,2024-01-02 06:00:00,Electronics,4.0,285.93,North,Card
6,1007,John Smith,2024-01-02 12:00:00,Clothing,6.0,346.29,East,Card
7,1008,John Smith,2024-01-02 18:00:00,Grocery,6.0,401.29,North,NaN
...,...,...,...,...,...,...,...,...
510,1323,Ali Khan,2024-03-21 12:00:00,Electronics,4.0,272.31,East,Card
511,1169,Fatima Noor,2024-02-12 00:00:00,Grocery,2.0,465.82,East,Cash
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West,Cash
513,1067,Sara Ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South,Card


In [68]:
# Q8. Filter rows by condition
# Get all orders where quantity is greater than 5.
df[df["quantity"] >5]

,order_id,customer,order_date,category,quantity,price,region,payment_method
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East,Cash
6,1007,John Smith,2024-01-02 12:00:00,Clothing,6.0,346.29,East,Card
7,1008,John Smith,2024-01-02 18:00:00,Grocery,6.0,401.29,North,NaN
8,1009,John Smith,2024-01-03 00:00:00,Clothing,8.0,104.07,West,NaN
10,1011,Fatima Noor,2024-01-03 12:00:00,Grocery,7.0,NaN,North,Online
...,...,...,...,...,...,...,...,...
507,1032,Sara Ahmed,2024-01-08 18:00:00,Grocery,8.0,45.41,East,Card
508,1250,Fatima Noor,2024-03-03 06:00:00,Grocery,7.0,94.67,East,Card
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West,Cash
513,1067,Sara Ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South,Card


In [70]:
# Q9. Sort and get the top N
# Sort df by unit_price descending and return the top 10 most expensive orders
df.sort_values(by="price",ascending=False).head(10)

,order_id,customer,order_date,category,quantity,price,region,payment_method
177,1178,Ali Khan,2024-02-14 06:00:00,Electronics,9.0,499.71,North,Cash
46,1047,Sara Ahmed,2024-01-12 12:00:00,Grocery,3.0,498.19,South,Card
428,1429,Ali Khan,2024-04-17 00:00:00,Electronics,5.0,497.46,East,Card
329,1330,Ali Khan,2024-03-23 06:00:00,Toys,3.0,497.10,North,NaN
351,1352,Fatima Noor,2024-03-28 18:00:00,Electronics,8.0,496.28,West,Card
377,1378,Ali Khan,2024-04-04 06:00:00,Grocery,1.0,496.12,East,Card
297,1298,Fatima Noor,2024-03-15 06:00:00,Clothing,9.0,493.95,East,Card
137,1138,Ali Khan,2024-02-04 06:00:00,Clothing,1.0,493.38,North,Card
179,1180,Ali Khan,2024-02-14 18:00:00,Clothing,9.0,488.70,North,NaN
467,1468,Sara Ahmed,2024-04-26 18:00:00,Electronics,5.0,488.06,West,Cash


In [71]:
# Q10. Select and drop columns
# Create a new DataFrame with only order_id , category , and unit_price . Separately, drop
# the payment_method column from a copy of df .
new_df = df[["order_id","category","price"]]
new_df


,order_id,category,price
0,1001,Electronics,178.56
2,1003,Clothing,43.90
5,1006,Electronics,285.93
6,1007,Clothing,346.29
7,1008,Grocery,401.29
...,...,...,...
510,1323,Electronics,272.31
511,1169,Grocery,465.82
512,1120,Grocery,422.86
513,1067,Toys,364.00


In [73]:
copy = df.copy()

In [75]:
copy.drop(columns=["payment_method"])

,order_id,customer,order_date,category,quantity,price,region
0,1001,Fatima Noor,2024-01-01 00:00:00,Electronics,7.0,178.56,East
2,1003,John Smith,2024-01-01 12:00:00,Clothing,4.0,43.90,West
5,1006,Sara Ahmed,2024-01-02 06:00:00,Electronics,4.0,285.93,North
6,1007,John Smith,2024-01-02 12:00:00,Clothing,6.0,346.29,East
7,1008,John Smith,2024-01-02 18:00:00,Grocery,6.0,401.29,North
...,...,...,...,...,...,...,...
510,1323,Ali Khan,2024-03-21 12:00:00,Electronics,4.0,272.31,East
511,1169,Fatima Noor,2024-02-12 00:00:00,Grocery,2.0,465.82,East
512,1120,Fatima Noor,2024-01-30 18:00:00,Grocery,9.0,422.86,West
513,1067,Sara Ahmed,2024-01-17 12:00:00,Toys,7.0,364.00,South


In [83]:
# Q11. Value counts
# Get the frequency count of each category and each region in df . Which category has the
# most missing values?
df["category"].value_counts(dropna=False)

category
Electronics    132
Grocery        111
Clothing       102
Toys            68
NaN             25
Name: count, dtype: int64

In [84]:
df["region"].value_counts()

region
West     121
East     108
North    105
South    104
Name: count, dtype: int64